# Disciplina 2: Analise de Dados
## Academia Vertice Fit | Projeto em Gestao de Sistemas Computacionais

Este notebook executa o pipeline analitico completo do projeto no Google Colab.

O codigo nao e reescrito aqui: o notebook importa os modulos do projeto e os
executa. Duplicar a logica entre o pacote e o notebook produziria duas versoes
que divergem com o tempo, e a versao apresentada na banca deixaria de ser a
versao testada.

**Estrutura da execucao**

| Secao | Conteudo | Requisito |
|-------|----------|-----------|
| 1 | Preparacao do ambiente | n/a |
| 2 | Extracao e inspecao da base bruta | 2.1 |
| 3 | Tratamento de dados (ETL) e trilha de auditoria | 2.1 |
| 4 | Indicadores e justificativa | 2.1 |
| 5 | Metodos estatisticos essenciais | 2.2 |
| 6 | Visualizacao de dados | 2.3 |
| 7 | Analise critica | 2.4 |

---
## 1. Preparacao do ambiente

O projeto e obtido e as dependencias sao instaladas. Caso os arquivos ja
estejam presentes na sessao, a celula apenas ajusta o diretorio de trabalho.

In [ ]:
# Preparacao do ambiente no Google Colab.
#
# A celula e idempotente: obtem o repositorio caso ele ainda nao esteja na
# sessao e apenas ajusta o diretorio de trabalho caso ja esteja. Executa-la
# duas vezes nao produz efeito diferente de executa-la uma vez.

import os
import subprocess
import sys
from pathlib import Path

REPOSITORIO = "https://github.com/warlyson30/projeto-gestao-sistemas.git"
RAIZ_REPO = Path("/content/projeto-gestao-sistemas")
RAIZ_PROJETO = RAIZ_REPO / "disciplina-2-analise-de-dados"


def projeto_valido(raiz: Path) -> bool:
    """O diretorio so serve se contiver os pacotes que este notebook importa."""
    return (raiz / "config" / "settings.py").exists() and (raiz / "src" / "pipeline.py").exists()


if not projeto_valido(RAIZ_PROJETO):
    print("Projeto ausente em", RAIZ_PROJETO, "- clonando de", REPOSITORIO)

    # O clone nunca e executado de dentro do diretorio de destino: remover o
    # proprio diretorio de trabalho deixa o processo com um cwd inexistente, e
    # todo comando de shell seguinte falha com "Unable to read current working
    # directory", erro cuja causa fica distante do sintoma.
    os.chdir("/content")
    subprocess.run(["rm", "-rf", str(RAIZ_REPO)], check=False)

    clone = subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORIO, str(RAIZ_REPO)],
        capture_output=True,
        text=True,
    )
    if clone.returncode != 0:
        print(clone.stderr.strip())
        raise RuntimeError(
            f"Falha ao clonar {REPOSITORIO}. Verifique a URL e o acesso ao repositorio."
        )

# A verificacao e refeita depois do clone: o repositorio pode existir sem
# conter a entrega. O erro precisa nomear essa causa, e nao a ausencia de
# permissao ou de rede, que produziriam o mesmo sintoma.
if not projeto_valido(RAIZ_PROJETO):
    raise RuntimeError(
        f"O clone de {REPOSITORIO} nao contem a pasta {RAIZ_PROJETO.name} "
        f"com config/ e src/. Causa provavel: o commit da Disciplina 2 ainda "
        f"nao foi enviado ao GitHub. Publique-o com 'git push origin main' e "
        f"execute esta celula novamente."
    )

os.chdir(RAIZ_PROJETO)
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

# O Colab ja fornece pandas, numpy, scipy, matplotlib e seaborn em versoes
# compativeis com o requirements.txt. Instalar apenas o que falta evita o
# aviso de reinicio de sessao, que descartaria o os.chdir e o sys.path
# definidos acima e faria o notebook falhar a partir da proxima celula.
try:
    import pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=False)

print("Diretorio de trabalho:", Path.cwd())
print("Conteudo:", ", ".join(sorted(p.name for p in RAIZ_PROJETO.iterdir() if p.name[0] != ".")))

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# O filtro e restrito ao ruido conhecido de renderizacao. Suprimir todos os
# avisos esconderia tambem os do pandas, do numpy e do scipy, que sinalizam
# mudanca de comportamento numerico: exatamente a classe de aviso que invalida
# uma analise sem produzir erro visivel.
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")
warnings.filterwarnings("ignore", category=UserWarning, module="seaborn")

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

from config.settings import BUSINESS, SMART_OBJECTIVES, ensure_directories

ensure_directories()

print("Ambiente pronto.")
print(f"pandas {pd.__version__} | numpy {np.__version__}")
print()
print("Objetivos SMART declarados no TAP:")
for codigo, objetivo in SMART_OBJECTIVES.items():
    print(f"  {codigo}: {objetivo.meta_numerica} ({objetivo.prazo})")

---
## 2. Extracao e inspecao da base bruta

A base representa a extracao do sistema de gestao academico. Ela **nao** esta
limpa: carrega os mesmos defeitos que ocorrem em sistemas transacionais reais,
originados de digitacao livre na recepcao, integracoes parciais e migracao de
sistema legado.

A inspecao desses defeitos antes do tratamento e o que permite justificar cada
regra de limpeza aplicada na secao seguinte.

In [ ]:
from src.etl.extract import extrair

bruta = extrair(forcar_regeracao=True)

print(f"Dimensoes: {bruta.shape[0]} registros x {bruta.shape[1]} colunas")
bruta.head(8)

In [ ]:
# Diagnostico dos defeitos presentes na origem

print("VALORES AUSENTES POR COLUNA")
ausentes = bruta.isna().sum()
ausentes = ausentes[ausentes > 0].sort_values(ascending=False)
for coluna, quantidade in ausentes.items():
    print(f"  {coluna:24s} {quantidade:5d}  ({quantidade / len(bruta) * 100:5.2f}%)")

print()
print("DUPLICATAS")
print(f"  Linhas integralmente duplicadas: {bruta.duplicated().sum()}")
print(f"  Chave de negocio (id_aluno) repetida: {bruta['id_aluno'].duplicated().sum()}")

print()
print("INCONSISTENCIA DE CATEGORIA (digitacao livre)")
print(f"  Valores distintos em 'plano': {sorted(bruta['plano'].dropna().unique().tolist())}")
print(f"  Categorias validas: {list(BUSINESS.planos_validos)}")

print()
print("VALORES FORA DE DOMINIO")
mensalidade = pd.to_numeric(bruta["valor_mensalidade"], errors="coerce")
idade = pd.to_numeric(bruta["idade"], errors="coerce")
print(f"  Mensalidade negativa: {(mensalidade < 0).sum()}")
print(f"  Idade fora de [{BUSINESS.idade_min}, {BUSINESS.idade_max}]: "
      f"{((idade < BUSINESS.idade_min) | (idade > BUSINESS.idade_max)).sum()}")

print()
print("FORMATOS DE DATA MISTOS")
print(f"  Amostra de data_matricula: {bruta['data_matricula'].dropna().unique()[:6].tolist()}")

---
## 3. Tratamento de dados (ETL): requisito 2.1

### 3.1 Ordem das etapas

As sete etapas de transformacao **nao** sao operacoes independentes. Inverter
qualquer uma delas altera o resultado numerico final:

1. **Normalizacao** precede a deduplicacao. Sem padronizar `"mensal"`,
   `"MENSAL"` e `" Mensal "` para uma unica forma, a deduplicacao trataria o
   mesmo registro como tres distintos.
2. **Deduplicacao** precede a imputacao. Registros replicados enviesariam a
   mediana e a moda usadas no preenchimento.
3. **Validacao de dominio** precede a imputacao. Uma idade registrada como 999
   contaminaria a mediana se fosse anulada apenas depois.
4. **Descarte de irrecuperaveis** remove registros sem os campos que definem a
   existencia comercial do contrato.
5. **Imputacao** com estrategia declarada por coluna.
6. **Deteccao de outliers**, em duas passagens de finalidade distinta.
7. **Engenharia de atributos**, que depende de todas as anteriores.

### 3.2 Estrategias de imputacao adotadas

| Coluna | Estrategia | Justificativa |
|--------|-----------|---------------|
| `frequencia_semanal`, `idade` | Mediana | Variaveis continuas assimetricas. A mediana e robusta a extremos; a media seria deslocada pelos proprios outliers que a analise investiga. |
| `modalidade_principal`, `unidade` | Moda | Variaveis categoricas nominais. A moda e a unica medida de tendencia central definida nesta escala. |
| `checkins_app_mes`, `usa_app` | Constante | A ausencia possui significado de negocio conhecido: nenhum check-in realizado, e nao informacao desconhecida. |

### 3.3 Tratamento de outliers

Os extremos sao **sinalizados**, nunca removidos. O criterio de Tukey (IQR) e
preferido ao z-score por nao pressupor normalidade e por seus limites nao serem
influenciados pelos proprios extremos que se pretende identificar.

Em uma academia, o aluno de ticket ou frequencia extrema e o ativo comercial
mais valioso da carteira: exclui-lo suprimiria justamente a evidencia que
sustenta a analise de concentracao de receita.

In [ ]:
from src.etl.transform import transformar
from src.etl.validate import QualityLedger, perfilar

ledger = QualityLedger()
ledger.perfil_entrada = perfilar(bruta, "base_bruta")

analitica = transformar(bruta, ledger)

ledger.perfil_saida = perfilar(analitica, "base_analitica")
ledger.imprimir_resumo()

In [ ]:
# Trilha de auditoria: cada tratamento com regra, volume e justificativa.
# Um pipeline que altera dados sem deixar rastro nao e auditavel.

trilha = pd.DataFrame(
    [
        {
            "Etapa": t.etapa,
            "Regra": t.regra,
            "Coluna": t.coluna[:40],
            "Afetados": t.registros_afetados,
            "Acao": t.acao[:48],
        }
        for t in ledger.tratamentos
    ]
)
trilha

In [ ]:
# Justificativa completa dos tratamentos que efetivamente alteraram dados

for tratamento in ledger.tratamentos:
    if tratamento.registros_afetados > 0:
        print(f"[{tratamento.etapa}] {tratamento.regra} -> {tratamento.coluna}")
        print(f"  Registros afetados: {tratamento.registros_afetados}")
        print(f"  Acao: {tratamento.acao}")
        print(f"  Justificativa: {tratamento.justificativa}")
        print()

In [ ]:
# Resultado do ETL: base analitica

print(f"Entrada:  {ledger.registros_entrada} registros")
print(f"Saida:    {ledger.registros_saida} registros")
print(f"Perda:    {ledger.registros_descartados} ({ledger.taxa_perda:.2%})")
print()
print("Ausentes remanescentes por coluna:")
restantes = analitica.isna().sum()
restantes = restantes[restantes > 0]
if restantes.empty:
    print("  Nenhum.")
else:
    for coluna, quantidade in restantes.items():
        print(f"  {coluna}: {quantidade}")
    print()
    print("  A ausencia de data_cancelamento nos alunos ativos e legitima:")
    print("  carrega o significado de contrato ainda vigente.")

print()
print(f"Colunas derivadas na engenharia de atributos: {analitica.shape[1] - bruta.shape[1]}")
analitica.head(6)

In [ ]:
from src.etl.load import carregar

arquivos = carregar(analitica)
for camada, caminho in arquivos.items():
    print(f"{camada:12s} {caminho}")

---
## 4. Indicadores e justificativa: requisito 2.1

Nenhum indicador deste catalogo e arbitrario. O padrao de registro adotado
torna **estruturalmente impossivel** acrescentar uma metrica ao relatorio sem
declarar simultaneamente sua formula, sua unidade, a direcao desejada, o
objetivo SMART do TAP que ela mede e a justificativa de sua existencia.

A restricao e deliberada: ela impede a proliferacao de metricas sem proposito,
problema recorrente em paineis gerenciais.

In [ ]:
from src.kpi.engine import calcular_kpis

kpis = calcular_kpis(analitica)
kpis.to_frame()

In [ ]:
# Justificativa formal de cada indicador e seu vinculo com o TAP

for resultado in kpis.resultados:
    print(f"{resultado.codigo} | {resultado.nome}")
    print(f"  Valor:      {resultado.valor_formatado}")
    print(f"  Formula:    {resultado.formula}")
    print(f"  Direcao:    {resultado.direcao_desejada}")
    print(f"  Objetivo:   {resultado.objetivo_smart} | {resultado.objetivo_meta}")
    print(f"  Por que existe: {resultado.justificativa}")
    print()

In [ ]:
# Hipotese central do projeto: adesao digital e evasao

hipotese = kpis.segmentacoes["hipotese_adesao_digital"]
print("ADESAO AO APLICATIVO E EVASAO")
print(f"  Evasao entre aderentes:     {hipotese['churn_com_app_pct']:.2f}%")
print(f"  Evasao entre nao aderentes: {hipotese['churn_sem_app_pct']:.2f}%")
print(f"  Diferenca:                  {hipotese['diferenca_pp']:.2f} pontos percentuais")
print(f"  Risco relativo:             {hipotese['risco_relativo']:.2f}x")

print()
risco = kpis.segmentacoes["carteira_de_risco"]
print("CARTEIRA DE RISCO PRIORITARIO (objetivo OE-02)")
print(f"  Alunos ativos:        {risco['alunos_ativos']}")
print(f"  Alunos em risco:      {risco['alunos_em_risco']} ({risco['participacao_pct']:.2f}%)")
print(f"  Receita mensal exposta: R$ {risco['mrr_em_risco']:,.2f}")

In [ ]:
# Segmentacoes: uma taxa global nao indica onde intervir; a taxa por
# segmento aponta o publico especifico da acao.

for nome in ("por_segmento_engajamento", "por_plano", "por_faixa_etaria", "por_unidade"):
    print(nome.replace("_", " ").upper())
    print(pd.DataFrame(kpis.segmentacoes[nome]).T.to_string())
    print()

---
## 5. Metodos estatisticos essenciais: requisito 2.2

Tres blocos, encadeados de forma nao decorativa.

**Decisao metodologica central:** o teste de normalidade nao e uma formalidade.
Seu resultado **seleciona** a familia de testes aplicada em seguida. Sob
normalidade, Pearson e o t de Student; caso contrario, Spearman e Mann-Whitney
passam a ser o resultado de referencia, e os testes parametricos permanecem
apenas para comparacao.

Reportar Pearson sobre variaveis comprovadamente nao normais seria erro
metodologico, ainda que o numero produzido parecesse plausivel.

**Segunda decisao:** todo valor-p vem acompanhado de tamanho de efeito (d de
Cohen, V de Cramer). Com amostras de milhares de registros, diferencas
irrelevantes para o negocio atingem significancia estatistica com facilidade; a
significancia isolada nao distingue diferenca relevante de diferenca trivial.

In [ ]:
from src.stats_engine.plan import HIPOTESES, VARIAVEIS_CONTINUAS, executar_analise

# As hipoteses sao declaradas ANTES da execucao. Isso evita percorrer todas as
# combinacoes de variaveis e reportar apenas as significativas, procedimento
# que inflaciona artificialmente a taxa de falsos positivos.

for hipotese in HIPOTESES:
    print(f"{hipotese.codigo} ({hipotese.objetivo_smart}): {hipotese.enunciado}")
    print(f"     Metodo: {hipotese.metodo}")
    print()

estatisticas = executar_analise(analitica)

### 5.1 Estatistica descritiva

In [ ]:
descritiva = pd.DataFrame(
    [
        {
            "Variavel": d.variavel,
            "n": d.n,
            "Media": d.media,
            "Mediana": d.mediana,
            "Desvio padrao": d.desvio_padrao,
            "CV (%)": d.coeficiente_variacao,
            "Min": d.minimo,
            "Q1": d.q1,
            "Q3": d.q3,
            "Max": d.maximo,
        }
        for d in estatisticas.descritivas
    ]
)
display(descritiva)

print("Leitura do coeficiente de variacao:")
print("O CV e adimensional, o que permite comparar a dispersao de variaveis")
print("medidas em unidades diferentes, algo que o desvio padrao nao possibilita.")
print()
for d in estatisticas.descritivas:
    print(f"  {d.variavel:22s} CV={d.coeficiente_variacao:6.1f}%  {d.interpretacao_estabilidade}")

### 5.2 Distribuicao e normalidade

In [ ]:
distribuicao = pd.DataFrame(
    [
        {
            "Variavel": d.variavel,
            "Assimetria": d.assimetria,
            "Curtose": d.curtose_excesso,
            "Teste": d.teste_normalidade,
            "p-valor": f"{d.p_valor:.2e}",
            "Normal": "Sim" if d.normal else "Nao",
            "Forma": d.interpretacao_assimetria,
        }
        for d in estatisticas.distribuicoes
    ]
)
display(distribuicao)

print("Implicacao metodologica de cada resultado:")
for d in estatisticas.distribuicoes:
    print(f"  {d.variavel}: {d.implicacao_metodologica}")

### 5.3 Correlacao e associacao

In [ ]:
correlacao = pd.DataFrame(
    [
        {
            "X": c.variavel_x,
            "Y": c.variavel_y,
            "Metodo": c.metodo,
            "Coeficiente": c.coeficiente,
            "p-valor": f"{c.p_valor:.2e}",
            "n": c.n,
            "Significativo": "Sim" if c.significativo else "Nao",
            "Forca": c.forca,
        }
        for c in estatisticas.correlacoes
    ]
)
display(correlacao)

In [ ]:
# Comparacao de grupos, com teste selecionado pela normalidade e
# tamanho de efeito sempre reportado.

for c in estatisticas.comparacoes:
    print(f"{c.variavel} agrupado por {c.agrupador}")
    print(f"  Teste aplicado: {c.teste}")
    print(f"  Media {c.grupo_a}: {c.media_a:.2f} (n={c.n_a})")
    print(f"  Media {c.grupo_b}: {c.media_b:.2f} (n={c.n_b})")
    print(f"  p-valor: {c.p_valor:.3e} | d de Cohen: {c.cohen_d:.3f} ({c.magnitude_efeito})")
    print()

In [ ]:
# Associacao entre variaveis categoricas: qui-quadrado com V de Cramer.
# O qui-quadrado cresce com o tamanho da amostra e nao mede intensidade;
# o V de Cramer normaliza a estatistica para o intervalo de 0 a 1.

associacao = pd.DataFrame(
    [
        {
            "Variavel A": a.variavel_a,
            "Variavel B": a.variavel_b,
            "Qui-quadrado": a.qui_quadrado,
            "gl": a.graus_liberdade,
            "p-valor": f"{a.p_valor:.2e}",
            "V de Cramer": a.cramers_v,
            "Significativo": "Sim" if a.significativo else "Nao",
            "Forca": a.forca,
        }
        for a in estatisticas.associacoes
    ]
)
display(associacao)

print("Achado negativo relevante:")
print("A unidade de matricula NAO apresentou associacao detectavel com a evasao.")
print("Isso descarta a hipotese de causa estrutural ou de infraestrutura fisica")
print("e reforca a natureza comportamental do gargalo identificado.")

In [ ]:
# Deteccao automatica de colinearidade.
# A verificacao e automatizada, e nao deixada a inspecao visual do mapa de
# calor, porque a colinearidade tem consequencia concreta: duas variaveis
# quase perfeitamente correlacionadas carregam a mesma informacao, e trata-las
# como evidencias independentes constitui dupla contagem.

colinearidades = estatisticas.matriz_correlacao.get("colinearidades", [])
if colinearidades:
    for achado in colinearidades:
        print(f"ALERTA: {achado['variavel_a']} x {achado['variavel_b']} = {achado['coeficiente']}")
        print(f"  {achado['diagnostico']}")
        print()
else:
    print("Nenhuma colinearidade acima do limite configurado.")

---
## 6. Visualizacao de dados: requisito 2.3

Das dez opcoes previstas no guia, **oito foram selecionadas e duas
descartadas**, com criterio declarado.

A selecao e deliberada e nao exaustiva. Um painel que reproduz todos os tipos
de grafico disponiveis demonstra dominio da biblioteca, nao dominio do
problema. Cada grafico aqui responde a uma pergunta especifica derivada das
hipoteses do plano de analise; os dois tipos descartados nao respondiam a
nenhuma.

In [ ]:
from src.viz.charts import JUSTIFICATIVA_SELECAO

for nome, justificativa in JUSTIFICATIVA_SELECAO.items():
    rotulo = nome.replace("descartado_", "DESCARTADO: ").upper()
    print(f"{rotulo}")
    print(f"  {justificativa}")
    print()

In [ ]:
from config.settings import FIGURES_DIR
from src.viz import charts

charts.aplicar_tema()
figuras = charts.gerar_todos(analitica, VARIAVEIS_CONTINUAS, FIGURES_DIR)

print(f"\n{len(figuras)} artefatos gerados em {FIGURES_DIR}")

In [ ]:
# Exibicao dos artefatos gerados

from IPython.display import Image, display

for nome, caminho in figuras.items():
    print(f"\n{'=' * 78}")
    print(f"{nome.upper()}: {JUSTIFICATIVA_SELECAO.get(nome, '')}")
    print("=" * 78)
    display(Image(filename=str(caminho)))

### Nota sobre a analise de coorte

O grafico de linhas mede a evasao dentro de uma **janela fixa de 90 dias**, e
exclui as coortes que ainda nao completaram esse intervalo.

Sem esse controle, as coortes recentes exibiriam evasao artificialmente baixa
apenas por ainda nao terem tido tempo de evadir. O grafico sugeriria uma
melhoria que nao ocorreu, e a gestao poderia declarar sucesso sobre um
artefato de censura a direita.

---
## 7. Analise critica: requisito 2.4

### Pergunta obrigatoria

> Baseado nas correlacoes estatisticas, nos outliers encontrados e no principal
> KPI calculado, qual e o maior gargalo operacional da empresa escolhida?
> Explique como esses dados numericos justificam as metas e o escopo definidos
> no TAP, na Disciplina 1.

In [ ]:
# Consolidacao dos numeros que sustentam a resposta

print("=" * 78)
print("EVIDENCIA CONSOLIDADA")
print("=" * 78)

print("\n1. INDICADOR PRINCIPAL E SUA DECOMPOSICAO")
for codigo in ("KPI-01", "KPI-02", "KPI-07"):
    r = kpis.obter(codigo)
    print(f"   {codigo}  {r.nome:46s} {r.valor_formatado}")

print("\n2. CORRELACOES COM O DESFECHO DE EVASAO")
for c in estatisticas.correlacoes:
    if c.metodo == "Ponto-bisserial":
        print(f"   {c.variavel_y:22s} r={c.coeficiente:+.4f}  p={c.p_valor:.2e}  ({c.forca})")

print("\n3. TAMANHO DE EFEITO ENTRE GRUPOS")
for c in estatisticas.comparacoes:
    print(f"   {c.variavel:22s} por {c.agrupador:10s} d={c.cohen_d:+.3f} ({c.magnitude_efeito})")

print("\n4. OUTLIERS E CONCENTRACAO DE RECEITA")
n_out = int(analitica["outlier_qualquer"].sum())
out_ticket = analitica[analitica["outlier_valor_mensalidade"]]
peso = out_ticket["receita_acumulada"].sum() / analitica["receita_acumulada"].sum() * 100
print(f"   Registros extremos sinalizados: {n_out} ({n_out / len(analitica) * 100:.1f}% da base)")
print(f"   Contratos de ticket extremo: {len(out_ticket)} "
      f"({len(out_ticket) / len(analitica) * 100:.1f}% da base)")
print(f"   Peso desses contratos na receita: {peso:.1f}%")
print(f"   {kpis.obter('KPI-12').nome}: {kpis.obter('KPI-12').valor_formatado}")

print("\n5. EXPOSICAO FINANCEIRA (base para o orcamento de retencao)")
for codigo in ("KPI-04", "KPI-08", "KPI-09", "KPI-10"):
    r = kpis.obter(codigo)
    print(f"   {codigo}  {r.nome:46s} {r.valor_formatado}")

### Resposta

O maior gargalo operacional da Academia Vertice Fit **nao e a captacao de novos
alunos, e sim a incapacidade de converter matricula em permanencia durante o
primeiro trimestre de contrato**. O principal indicador calculado, a taxa de
evasao da carteira (KPI-01), atingiu **27,81%**, e sua decomposicao revela onde
o problema se concentra: **28,95% de todos os cancelamentos ocorrem nos
primeiros sessenta dias** (KPI-02), e a permanencia media do aluno que evade e
de apenas **3,59 meses** (KPI-07). A analise estatistica identificou a causa
comportamental dessa perda. A correlacao ponto-bisserial entre evasao e
frequencia semanal foi de **-0,347** (p = 5,7 x 10⁻⁴³), e a comparacao entre os
dois grupos pelo teste de Mann-Whitney confirmou diferenca com **tamanho de
efeito grande (d de Cohen = -0,82)**: o aluno retido comparece 3,78 vezes por
semana, contra 2,60 do aluno que cancela. O mesmo padrao aparece na dimensao
digital, com **risco relativo de 2,44**, ja que a evasao entre nao aderentes ao
aplicativo alcanca 47,40% contra 19,42% dos aderentes (qui-quadrado
p = 7,7 x 10⁻²⁸, V de Cramer = 0,28). O segmento de engajamento sintetiza os
dois fatores e apresenta a associacao mais forte de toda a analise
(V de Cramer = 0,37): a evasao varia de **12,89% no segmento de alto
engajamento a 56,58% no de baixo engajamento**. Cabe registrar um achado
negativo relevante: a unidade de matricula nao apresentou associacao detectavel
com a evasao (p = 0,505), o que descarta a hipotese de causa estrutural ou de
infraestrutura fisica e reforca a natureza comportamental do gargalo.

A analise de dispersao e de outliers acrescenta a dimensao financeira do
problema e **altera a prioridade da acao**. O coeficiente de variacao da receita
acumulada por aluno e de 90,8%, indicando processo altamente instavel, e o
criterio de Tukey sinalizou 68 registros extremos, dos quais 42 correspondem a
contratos com acompanhamento personalizado. Esses 42 alunos, que representam
2,8% da carteira, **respondem por 10,8% de toda a receita acumulada**; de forma
mais ampla, os 20% maiores contratos concentram **46,88% do faturamento**
(KPI-12). Como o valor da mensalidade nao apresentou associacao com a evasao
(r = -0,008, p = 0,750), o contrato de alto valor cancela na mesma proporcao que
o contrato padrao, porem com impacto financeiro varias vezes superior. Esses
extremos foram deliberadamente sinalizados e preservados na base, e nao
removidos: descarta-los teria eliminado justamente o segmento que sustenta a
concentracao de receita e distorcido para baixo toda a estimativa de perda. O
conjunto desses numeros justifica diretamente as metas e o escopo definidos no
TAP. A meta de reduzir o churn em dez pontos percentuais (OE-01) e realista
porque a diferenca observada entre segmentos, de 43,69 pontos, e muito superior
a ela, evidenciando margem de manobra. O recorte de carteira de risco (OE-02)
deixou de ser arbitrario e passou a ter criterio verificavel: os **122 alunos
ativos** que combinam ausencia de adesao digital e frequencia inferior a tres
dias semanais representam 11,43% da base e **R$ 14.426,65 de receita mensal em
risco** (KPI-08 e KPI-09), montante que estabelece o teto economico defensavel
para o investimento em retencao. A meta de elevar a adesao ao aplicativo para
80% (OE-03) e sustentada por ser esta a unica variavel do modelo diretamente
influenciavel pela empresa, ao contrario da frequencia, que depende da
disponibilidade do aluno. Por fim, a taxa de subutilizacao de **31,30%**
(KPI-10) quantifica a demanda reprimida que alimenta o modelo de otimizacao da
grade de horarios da Disciplina 4, assegurando continuidade metodologica entre
as etapas do projeto.

### Ressalvas metodologicas

A analise e **observacional e transversal**. Ela estabelece associacao
estatistica, nao causalidade, e permanece sujeita a causalidade reversa: o aluno
pode reduzir a frequencia por ja ter decidido cancelar, em vez de cancelar por
frequentar pouco. A confirmacao causal exigiria desenho experimental com grupo
de controle, recomendado como escopo de etapa posterior.

Duas restricoes tecnicas foram identificadas e tratadas explicitamente: a
colinearidade quase perfeita entre tempo de vinculo e receita acumulada
(Spearman = 0,97), detectada automaticamente pelo pipeline; e a censura a
direita na analise de coortes, corrigida pela janela fixa de noventa dias.

Todas as seis variaveis continuas rejeitaram a normalidade. Por essa razao,
Spearman e Mann-Whitney foram adotados como resultado de referencia, e os testes
parametricos equivalentes permanecem reportados apenas para comparacao.

---
## 8. Execucao integrada e verificacao

As celulas anteriores executam o pipeline etapa a etapa, para fins didaticos.
Em operacao, todo o processo e disparado por um unico comando.

In [ ]:
# Execucao integrada equivalente a linha de comando: python main.py

from src.pipeline import executar_pipeline

resultado = executar_pipeline(forcar_regeracao=False, gerar_figuras=False)

print()
for chave, valor in resultado.resumo().items():
    print(f"  {chave.replace('_', ' ').capitalize():34s} {valor}")

In [ ]:
# Suite de testes automatizados

!pip install -q pytest
!python -m pytest -q

---

## Bibliotecas utilizadas

| Biblioteca | Uso no projeto |
|-----------|----------------|
| `pandas` | Manipulacao tabular, agregacoes e series temporais |
| `numpy` | Operacoes vetorizadas e geracao pseudoaleatoria |
| `scipy.stats` | Testes de normalidade, correlacao, Mann-Whitney e qui-quadrado |
| `matplotlib` | Construcao dos artefatos graficos |
| `seaborn` | Mapa de calor, grafico de violino e tema visual |
| `pyarrow` | Camada intermediaria em Parquet com preservacao de tipos |
| `pytest` | Suite de testes automatizados |

Todo o codigo executa sem erro no Google Colab. A ausencia de `pyarrow` nao
interrompe o pipeline: a camada intermediaria degrada para CSV com aviso
registrado em log.